In [1]:
%load_ext autoreload
%autoreload 2

# Author

> Author Class and all methods to get details of each author form the publication

In [2]:
#| default_exp author

In [3]:
#| hide
from nbdev.showdoc import *

In [4]:
#| export
from Bio import Entrez
import sys
# from tinydb import TinyDB, Query, where
import pandas as pd
from datetime import datetime, timedelta, date
from collections import defaultdict, Counter
import  pickle
from fastcore.all import *
from dotenv import load_dotenv
from pydantic import BaseModel, field_validator, ValidationError, ConfigDict
from typing import Any

/var/folders/4_/_3_qxk816cq05p7xgwhzqmnm0000gn/T/ipykernel_51104/2837552307.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [5]:
#| exports
class Autor(BaseModel):
    model_config = ConfigDict(extra='allow')
    Fname:str 
    Lname:str 
    name:str 
    initials:str
    emails:str
    affiliations:str
    identifier:str | None
    affiliation_parsed: Any
    
    @field_validator('name', 'Fname', 'Lname', 'initials', 'affiliations')
    def validate_str(cls, v):
        return v.title()
    
    @field_validator('identifier', mode='before')
    @classmethod
    def validate_identifier(cls, v):
        if not v:
            return ''
        else:
            return v

In [6]:
data = {
    'Fname': 'autorFN',
    'Lname': 'autorLN',
    'emails': 'emails',
    'affiliations': 'AFFs', 
    'identifier': 'autorID',
    'name': 'name', 
    'initials': 'autorIN',
    'affiliation_parsed':{}
       }  #

In [10]:
Autor(**data)

Autor(Fname='Autorfn', Lname='Autorln', name='Name', initials='Autorin', emails='emails', affiliations='Affs', identifier='autorID', affiliation_parsed={})

In [11]:
x = Autor(**data)

In [12]:
x.model_dump()

{'Fname': 'Autorfn',
 'Lname': 'Autorln',
 'name': 'Name',
 'initials': 'Autorin',
 'emails': 'emails',
 'affiliations': 'Affs',
 'identifier': 'autorID',
 'affiliation_parsed': {}}

In [ ]:
#| hide
@patch
def add_db(
    self:Autor
)->bool:
    try:
        if self.check_db():
            self.update_db()
        else:
            print('add to database')
            db = TinyDB(self.db_name)
            #updated = date.today()
            #self.updated = updated.strftime('%d-%m-%Y')
            db.insert(self.to_dict())
            db.close()
        return True
    except:
        print('Error adding autor to the database')
        return False

@patch
def search(self:Autor):
    db = TinyDB(self.db_name)#, storage=serialization)
    query = Query()
    result = db.search((query.Lname.matches(self.name)) | (query.Fname.matches(self.name)) | (query.name.matches(self.name)))
    if len(result) > 0:
        db.close()
        return result
    else:
        db.close()
        return 

@patch
def check_db(self:Autor):
    #if exist return True, else False
    db = TinyDB(self.db_name)#, storage=serialization)
    query = Query()
    if db.search(query.name == self.name):
        db.close()
        return True
    else:
        db.close()
        return False

@patch
def update_papers(self:Autor):
    db = TinyDB(self.db_name)
    query = Query()
    db.update({'n_papers':self.n_papers},
              query.name == self.name)
    db.close()

@patch
def update_db(self:Autor):
    db = TinyDB(self.db_name)
    query = Query()
    data = db.get(query.name == self.name)

    autor = self.merge_autors(Autor(data))
    db.update(autor.to_dict(), query.name == self.name)
    db.close()

def merge_autors(self, autor):
    #merge all variables from 2 autors.
    autor_dict = self.to_dict()
    tmp_dict = autor.to_dict()

    autor_dict['emails'].extend(tmp_dict['emails'])
    autor_dict['countries'].extend(tmp_dict['countries'])
    autor_dict['identifier'].extend(tmp_dict['identifier'])

    autor_dict['n_papers'] = max([autor_dict['n_papers'], tmp_dict['n_papers']])

    autor_dict['affiliations'] = ';'.join([autor_dict['affiliations'], tmp_dict['affiliations']])
    autor_dict['state'] = ';'.join([autor_dict['state'], tmp_dict['state']])

    autor_dict['affiliations'] = ';'.join(list(set(autor_dict['affiliations'].split(';'))))
    autor_dict['emails'] = list(set(autor_dict['emails']))
    autor_dict['countries'] = list(set(autor_dict['countries']))
    autor_dict['identifier'] = list(set(autor_dict['identifier']))

    return Autor(autor_dict)

def to_dict(self):

    parser = {'Fname': self.Fname, 'Lname': self.Lname, 'emails': list(set(self.email)), 'affiliations': self.affiliations,
                    'countries': list(set(self.country)), 'identifier':list(set(self.id)), 'name':self.name,
              'updated':self.updated, 'n_papers':self.n_papers, 'state':self.state}
    return parser

In [13]:
import nbdev; nbdev.nbdev_export()